In [13]:
import asyncio
import os
from typing import Annotated
from dotenv import load_dotenv
from langchain_openrouter import ChatOpenRouter
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from psycopg_pool import AsyncConnectionPool
from typing_extensions import TypedDict

In [14]:
load_dotenv()

True

In [15]:
DATABASE_URL = os.getenv("DATABASE_URL")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

In [16]:
model = ChatOpenRouter(
    model="openrouter/free",
    api_key=OPENROUTER_API_KEY,
    temperature=0.7,
)

In [17]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [18]:
async def chatbot(state: ChatState):

    response = await model.ainvoke(state["messages"])

    return {
        "messages": [response]
    }

In [19]:
builder = StateGraph(ChatState)

In [20]:
builder.add_node("chatbot", chatbot)

In [21]:
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

In [22]:
async def main():

    async with AsyncConnectionPool(
        DATABASE_URL,
        min_size=4,
        max_size=10
    ) as pool:

        checkpointer = AsyncPostgresSaver(pool)
        await checkpointer.setup()

        graph = builder.compile(checkpointer=checkpointer)

        thread_id = "chat_2"

        print("\nType 'exit' to quit.\n")

        while True:

            query = input("You : ")

            if query.lower() == "exit":
                break

            result = await graph.ainvoke({"messages": [HumanMessage(content= query)]},
                config={"configurable": {"thread_id": thread_id,}},)

            result = "\nAssistant :", result["messages"][-1].content
            display(Markdown(f"{result[0]} {result[1]}"))

In [23]:
from IPython.display import display, Markdown

In [5]:
from rag_with_chroma_db import my_rag

In [32]:
rag_response = my_rag.retriever("total steps to write it?", 3, user_id = "user_123")

In [33]:
print(rag_response[0].page_content)

USE BULLET POINTS:
Present information in bullet points to
make it easily scannable. Start each
bullet point with a strong action verb.
@nehamalhotra


In [25]:
await main()


Type 'exit' to quit.




Assistant : The extracted information from the uploaded documents provides the following details:  

### **Page References**:  
1. **Document 1 (Page 1)**: Contains mandatory sections like contact info, education, work experience, summary, etc.  
2. **Document 2 (Page 6)**: Contains **bonus tips** (fonts, formatting, etc.).  
3. **Document 3 (Source Document)**: Likely contains the full template or structure (e.g., "MY WINNING RESUME" template).  

### **Key Info Found**:  
- **Page 1**: Refers to the **mandatory sections** listed in the first document.  
- **Page 6**: Includes **bonus formatting guidelines** (fonts, conciseness, etc.).  
- **Document 3**: The root source (e.g., a template or example for structuring resumes).  

### **How to Locate It**:  
- In the first document, focus on sections like **contact info**, **education**, and **work experience**.  
- In the second document, locate the **"Bonus Tips"** section (specifically page 6).  
- For the third document, refer to it directly (e.g., "MY WINNING RESUME" template).  

If you need further clarification or help adapting the structure, let me know! 😊